In [1]:
import sys; sys.path.append("..")
import time, os
import pandas as pd

# WHY psutil: it reports this process's memory, so we measure only the index, not the whole laptop
try:
    import psutil
except ImportError:
    raise SystemExit("run: pip install psutil   (then re-run this cell)")

from src.blocking import build_index, prepare_pool

D = "../dataset/dev/"
pool_df = pd.concat([pd.read_csv(D + "s2_dev.tsv", sep="\t"),
                     pd.read_csv(D + "s3_dev.tsv", sep="\t")], ignore_index=True)

proc = psutil.Process(os.getpid())
mb = lambda b: round(b / 1e6)

# WHY measure before: the pool DataFrame itself is already in memory, so only the growth counts
print("free RAM now (MB):", mb(psutil.virtual_memory().available))
m0 = proc.memory_info().rss

t0 = time.time()
index = build_index(pool_df)
m1 = proc.memory_info().rss
prepared = prepare_pool(pool_df)
m2 = proc.memory_info().rss

print("index: keys", len(index), "| seconds", round(time.time() - t0, 1))
print("index memory (MB):", mb(m1 - m0))
print("prepared dict memory (MB):", mb(m2 - m1))

# WHY scale by records: 10.3M full pool records against 467,494 dev records
factor = 10_320_219 / len(pool_df)
print("scale factor:", round(factor, 1))
print("rough full-scale index + prepared (GB):", round((m2 - m0) * factor / 1e9, 1))

free RAM now (MB): 9017
index: keys 2259243 | seconds 23.7
index memory (MB): 465
prepared dict memory (MB): 587
scale factor: 22.1
rough full-scale index + prepared (GB): 23.2


In [3]:
import pandas as pd
from pathlib import Path

T = Path("../dataset/train")
for f in ["train_source1.tsv", "train_source2.tsv", "train_source3.tsv"]:
    # WHY usecols: loads one column only, so this is safe even with little free RAM
    counts = pd.read_csv(T / f, sep="\t", usecols=["country"])["country"].value_counts()
    print(f, counts.to_dict())

train_source1.tsv {'US': 1323633, 'India': 883188}
train_source2.tsv {'US': 3016817, 'India': 2017799}
train_source3.tsv {'US': 3170056, 'India': 2115547}
